# Cohere integration: every backend, one table

The same Cohere models go through every way AuditKIT can reach a model, with the same checks each time. The notebook ends with a PASS / FAIL / SKIP table: one row per backend and check.

| § | Backend | Cohere models | Checks |
|---|---|---|---|
| 1 | `hf:`, transformers in this runtime | Tiny Aya, Command R7B | generation; tools: **native** for Command R7B (its template renders tools), **prompt mode** for Tiny Aya, and native **refused with advice** for Tiny Aya |
| 2 | `vllm:`, the offline vLLM engine | same | same |
| 3 | `vllm serve` → `api:` | same | generation; prompt-mode tools |
| 4 | SGLang server → `api:` | Tiny Aya; Command R7B with `--tool-call-parser cohere_command4` | generation; tools; Command R7B's calls must come back **structured** from SGLang's parser |
| 5 | Cohere hosted API → `api:` | Command R7B, Command A | generation; native tools |
| 6 | HF Inference Providers → `api:` | a Cohere model on the `cohere` provider | generation; tools |

The **checks** are the same everywhere:
- **Generation:** four multilingual questions, where `quasi_exact_match` must be at least 0.5.
- **Tools:** single, parallel and no-tool cases, where `tool_call_f1` must be above 0 and `tool_call_validity` must be 1.0.
- **Refusal:** asking Tiny Aya for native tools must raise `CapabilityError`, never silently drop the tools.

**Runtime:** a Colab **A100** (40 or 80 GB). An L4 runs everything except Command R7B, which needs 24 GB or more. §1–§4 run one model at a time and free the GPU between them.

**Tokens:** asked for in hidden boxes, never saved in the notebook.
- `GITHUB_TOKEN`: required, to install AuditKIT.
- `HF_TOKEN`: the gated models, and §6. The account must have accepted the Tiny Aya and Command R7B licences.
- `CO_API_KEY`: §5.

A section whose token or GPU is missing is SKIP, not FAIL.

**Time:** about 40–60 minutes on a fresh runtime, mostly installs and model downloads.

In [ ]:
# ---- settings ----
BRANCH = "main"   # SGLang launch advice (any GPU) is merged (#56)
TINY_AYA = "CohereLabs/tiny-aya-global"
COMMAND_R7B = "CohereLabs/c4ai-command-r7b-12-2024"
COHERE_API_MODELS = ["command-r7b-12-2024", "command-a-03-2025"]
COHERE_API_BASE = "https://api.cohere.ai/compatibility/v1"
HF_ROUTER_BASE = "https://router.huggingface.co/v1"
HF_ROUTER_MODEL = "CohereLabs/c4ai-command-r7b-12-2024:cohere"   # model:provider; the probe in §6 skips it if unavailable
RUN = {"hf": True, "vllm": True, "vllm_serve": True, "sglang": True, "cohere_api": True, "hf_router": True}
PORT = 30000

In [ ]:
# ---- tokens: paste each one into the box that appears (hidden input; never saved in the notebook) ----
import os
from getpass import getpass
for key, why in (("GITHUB_TOKEN", "required: installs AuditKIT from the private repo"),
                 ("HF_TOKEN", "gated Cohere models and §6; Enter to skip"),
                 ("CO_API_KEY", "Cohere hosted API, §5; Enter to skip")):
    if os.environ.get(key):
        print(f"{key}: already set in this runtime")
        continue
    value = getpass(f"Paste {key} ({why}): ").strip()
    if value:
        os.environ[key] = value
print({k: bool(os.environ.get(k)) for k in ("GITHUB_TOKEN", "HF_TOKEN", "CO_API_KEY")})

In [ ]:
# ---- install AuditKIT with the hf: and vllm: backends (an existing clone is moved to the branch head) ----
import subprocess, sys
REPO = "/content/AuditKIT-Internal"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if "torch" in sys.modules:
        raise SystemExit("torch was imported before this install: Runtime -> Restart session, then run from the top.")
    assert os.environ.get("GITHUB_TOKEN"), "paste GITHUB_TOKEN in the cell above"
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, repo_url, REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "fetch", "-q", repo_url, BRANCH], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", f"{REPO}[transformers,vllm,requests]"], check=True)
    # vLLM installs a CUDA 13 torch; Colab's CUDA 12.8 torchaudio would then break every transformers import
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
    sys.path.insert(0, f"{REPO}/src")
try:
    from huggingface_hub import get_token, login
    if not os.environ.get("HF_TOKEN") and get_token():
        os.environ["HF_TOKEN"] = get_token()
    if os.environ.get("HF_TOKEN"):
        login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
except Exception as e:  # noqa: BLE001
    print("HF login skipped:", e)
import auditkit as ak, torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print("auditkit", ak.__version__, "| commit", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip() if IN_COLAB else "-",
      "| GPU", f"{GPU_GB:.0f} GB" if GPU_GB else "none")
print(ak.check_compat(check_cuda=True))

In [ ]:
# ---- the shared checks: every backend runs the same ones ----
import dataclasses, gc, json, time
import pandas as pd
from auditkit import Sample
from auditkit.errors import CapabilityError
from auditkit.runspec import RunConfig
RESULTS = []

def record(backend, model, check, outcome, detail=""):
    RESULTS.append({"backend": backend, "model": model, "check": check, "outcome": outcome, "detail": str(detail)[:160]})
    print(f"  {outcome:5s} {backend:12s} {model.split('/')[-1]:28s} {check:26s} {str(detail)[:90]}")

QA = [("What is the capital of France? Answer with one word.", "Paris"),
      ("Traduce al inglés, solo la traducción: gracias", "thank you"),
      ("Quelle est la capitale du Japon ? Un seul mot.", "Tokyo"),
      ("Wie viele Tage hat eine Woche? Nur die Zahl.", "7")]
GEN = [Sample(id=f"qa{i}", input=q, target=a) for i, (q, a) in enumerate(QA)]
TOOLS = [{"type": "function", "function": {"name": "get_weather", "description": "Current weather for a city",
          "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
         {"type": "function", "function": {"name": "get_time", "description": "Current local time in a city",
          "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}]
W = lambda c: {"name": "get_weather", "arguments": {"city": c}}
TOOL_TASKS = [
    Sample(id="single", input="What's the weather in Lisbon?", tools=TOOLS, expected_tool_calls=[[W("Lisbon")]]),
    Sample(id="parallel", input="What's the weather in Oslo and in Cairo? Check both.", tools=TOOLS, expected_tool_calls=[[W("Oslo"), W("Cairo")]]),
    Sample(id="no_tool", input="What is 2+2? Answer directly, without tools.", tools=TOOLS, expected_tool_calls=[]),
]
CFG = RunConfig(temperature=0.0, max_tokens=256, concurrency=1, max_retries=3, retry_delay=5)
HOSTED = {"max_retries": 6, "retry_delay": 10}   # hosted APIs: trial keys are rate-limited

def check_generation(backend, model, m, retries=None):
    try:
        r = ak.evaluate(GEN, model=m, scorers=["quasi_exact_match"], config=RunConfig(temperature=0.0, max_tokens=48, concurrency=1, **(retries or {})))
        qem = r.headline.get("quasi_exact_match")
        ok = qem is not None and qem >= 0.5 and not r.errors
        answers = [(p.raw_output or "").strip()[:20] for p in r.predictions]
        record(backend, model, "generation", "PASS" if ok else "FAIL", f"qem={qem} errors={len(r.errors)} {answers}")
    except Exception as e:  # noqa: BLE001
        record(backend, model, "generation", "FAIL", f"{type(e).__name__}: {e}")

def check_tools(backend, model, m, mode, require_structured=False, retries=None):
    label = f"tools ({mode})" + (" structured" if require_structured else "")
    try:
        r = ak.evaluate(TOOL_TASKS, model=m, adapter=ak.ToolCallAdapter(mode=mode),
                        scorers=[ak.ToolCallF1(), ak.ToolCallValidity(), ak.ParallelToolCalls()],
                        config=dataclasses.replace(CFG, **retries) if retries else CFG)
        f1, valid = r.headline.get("tool_call_f1"), r.headline.get("tool_call_validity", 1.0)
        structured = sum(bool(((p.context or {}).get("trace") or {}).get("tool_calls")) for p in r.predictions)
        ok = f1 is not None and f1 > 0 and valid == 1.0 and not r.errors and (structured > 0 or not require_structured)
        record(backend, model, label, "PASS" if ok else "FAIL",
               f"f1={f1} validity={valid} structured={structured} errors={len(r.errors)} "
               f"{[(p.sample_id, (p.raw_output or '')[:60]) for p in r.predictions]}")
    except Exception as e:  # noqa: BLE001
        record(backend, model, label, "FAIL", f"{type(e).__name__}: {e}")

def check_native_refused(backend, model, m):
    try:
        ak.evaluate(TOOL_TASKS[:1], model=m, adapter=ak.ToolCallAdapter(), scorers=[ak.ToolCallF1()], config=CFG)
        record(backend, model, "native refused (no tool slot)", "FAIL", "native tools were accepted by a template without a tool slot")
    except CapabilityError as e:
        record(backend, model, "native refused (no tool slot)", "PASS", str(e)[:120])
    except Exception as e:  # noqa: BLE001
        record(backend, model, "native refused (no tool slot)", "FAIL", f"{type(e).__name__}: {e}")

def skip(backend, model, why):
    record(backend, model, "(all checks)", "SKIP", why)

def free(m=None):
    if m is not None and callable(getattr(m, "unload", None)):
        try:
            m.unload()
        except Exception:  # noqa: BLE001
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def free_gb():
    return torch.cuda.mem_get_info()[0] / 2**30 if torch.cuda.is_available() else 0.0

def need(model):
    """Why this model can't run here, or None."""
    if not os.environ.get("HF_TOKEN"):
        return "needs HF_TOKEN (gated model)"
    if model == COMMAND_R7B and GPU_GB < 23:
        return f"needs a 24+ GB GPU (this one: {GPU_GB:.0f} GB)"
    return None

## 1 · `hf:` (transformers in this runtime)

- **Tiny Aya**'s chat template has no tool slot, so native tools must be **refused** with a `CapabilityError`, and prompt mode used instead.
- **Command R7B**'s template renders tools and answers in Cohere's action format (`<|START_ACTION|>[…]`), which AuditKIT parses from the text.

In [ ]:
from auditkit.model.hf_gen import HFGenModel
if RUN["hf"]:
    for model in (TINY_AYA, COMMAND_R7B):
        why = need(model)
        if why:
            skip("hf:", model, why); continue
        m = HFGenModel(model, device="cuda", name=f"hf:{model}", token=os.environ["HF_TOKEN"])
        check_generation("hf:", model, m)
        if model == TINY_AYA:
            check_native_refused("hf:", model, m)
            check_tools("hf:", model, m, "prompt")
        else:
            check_tools("hf:", model, m, "native")
        del m; free()

## 2 · `vllm:` (the offline vLLM engine)

It's the same contract as `hf:`. The engine renders each model's own chat template, with `tools=` for native mode, and refuses native tools when the template has no tool slot.

In [ ]:
from auditkit.model.vllm_gen import VLLMModel
if RUN["vllm"]:
    DTYPE = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float16"
    for model in (TINY_AYA, COMMAND_R7B):
        why = need(model)
        if why:
            skip("vllm:", model, why); continue
        try:
            m = VLLMModel(model, name=f"vllm:{model}", dtype=DTYPE, gpu_memory_utilization=0.80, max_model_len=8192)
        except Exception as e:  # noqa: BLE001
            record("vllm:", model, "(engine load)", "FAIL", f"{type(e).__name__}: {e}"); continue
        check_generation("vllm:", model, m)
        if model == TINY_AYA:
            check_native_refused("vllm:", model, m)
            check_tools("vllm:", model, m, "prompt")
        else:
            check_tools("vllm:", model, m, "native")
        free(m); del m; free()
    # vllm: unload runs vLLM's own teardown; on 0.30 the engine's memory must come back
    # here, or the next section's `vllm serve` fails its free-memory check
    print(f"GPU free after the offline engine: {free_gb():.1f} of {GPU_GB:.1f} GB")

## 3 · `vllm serve` → `api:`

vLLM's OpenAI server runs in this runtime, and AuditKIT reaches it over HTTP, as it would a deployed server. Tools are checked in **prompt mode**, because no vLLM tool parser is configured here for the Cohere formats. AuditKIT parses the calls from the reply text.

In [ ]:
import re, requests, signal
from auditkit.model.api_gen import APIModel

def wait_healthy(proc, log, timeout_s=1200):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            return False
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5).ok:
                return True
        except requests.RequestException:
            pass
        time.sleep(10)
    return False

def start_failure(log):
    # why a server didn't come up: free GPU memory and the last error line go in the table (160 chars),
    # the log's last 30 lines go to the cell output
    lines = open(log).read().splitlines()
    print(f"---- {log} (last 30 lines) ----", *lines[-30:], sep="\n")
    errors = [l for l in lines if re.search(r"Error|error:|out of memory|Free memory", l)]
    return f"free GPU {free_gb():.1f} GB | " + (errors[-1].strip() if errors else lines[-1].strip() if lines else "empty log")

def stop(proc):
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM); proc.wait(timeout=120)
    except Exception:  # noqa: BLE001
        pass
    free()

def server_env():
    # importing vllm: sets VLLM_ENABLE_V1_MULTIPROCESSING=0 for its in-process engine;
    # a server is its own process and must not inherit that
    return {k: v for k, v in os.environ.items() if k != "VLLM_ENABLE_V1_MULTIPROCESSING"}

def served(model, name):
    return APIModel(model, api_base=f"http://127.0.0.1:{PORT}/v1", api_key="EMPTY", name=name)

if RUN["vllm_serve"]:
    for model in (TINY_AYA, COMMAND_R7B):
        why = need(model)
        if why:
            skip("vllm serve", model, why); continue
        log = f"/tmp/vllm_serve_{model.split('/')[-1]}.log"
        proc = subprocess.Popen([sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", model, "--port", str(PORT),
                                 "--gpu-memory-utilization", "0.80", "--max-model-len", "8192"],
                                stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True, env=server_env())
        try:
            if not wait_healthy(proc, log):
                record("vllm serve", model, "(server start)", "FAIL", start_failure(log)); continue
            m = served(model, f"api:vllm/{model}")
            check_generation("vllm serve", model, m)
            check_tools("vllm serve", model, m, "prompt")
        finally:
            stop(proc)

## 4 · SGLang server → `api:`

SGLang can't share an environment with vLLM, so it gets its own Python 3.12 environment, built with uv, with `ninja` for flashinfer's kernel builds. Command R7B is served with `--tool-call-parser cohere_command4`, and its calls must come back as **structured** `tool_calls`: the live check that Cohere's action markers reach SGLang's parser.

In [ ]:
if RUN["sglang"]:
    SGL_ENV = "/content/sglang-env" if os.path.isdir("/content") else os.path.expanduser("~/sglang-env")
    SGL_PY = f"{SGL_ENV}/bin/python"
    def env_py():
        if not os.path.exists(SGL_PY):
            return None
        return subprocess.run([SGL_PY, "-c", "import sys; print('%d.%d' % sys.version_info[:2])"], capture_output=True, text=True).stdout.strip()
    if env_py() != "3.12":
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", "--seed", "--clear", SGL_ENV], check=True)
    for spec in ("--upgrade pip", "sglang>=0.5.18,<0.6", "ninja"):
        subprocess.run([SGL_PY, "-m", "pip", "install", "-q", *spec.split()], check=True)
    # AuditKIT's advice for this GPU (DeepGEMM off on Hopper/Blackwell; fp16 without bf16), machine's own CUDA
    subprocess.run([SGL_PY, f"{REPO}/src/auditkit/compat.py", "--patch-sglang"], check=False)
    sgl_env = {**os.environ, "PATH": f"{SGL_ENV}/bin:" + os.environ.get("PATH", "")}
    if os.path.isdir("/usr/local/cuda"):
        sgl_env.setdefault("CUDA_HOME", "/usr/local/cuda")
    sgl_env.pop("VLLM_ENABLE_V1_MULTIPROCESSING", None)
    out = subprocess.run([SGL_PY, f"{REPO}/src/auditkit/compat.py", "--sglang-launch", "x"], capture_output=True,
                         text=True, env=sgl_env).stdout
    line = next((l for l in out.splitlines() if l.startswith("sglang launch advice:")), None)
    advice = json.loads(line.split(":", 1)[1]) if line else {"env": {}, "args": []}
    sgl_env.update(advice["env"])
    print("SGLang launch advice for this GPU:", advice)
    for model, extra, native in ((TINY_AYA, [], False), (COMMAND_R7B, ["--tool-call-parser", "cohere_command4"], True)):
        why = need(model)
        if why:
            skip("sglang", model, why); continue
        log = f"/tmp/sglang_{model.split('/')[-1]}.log"
        proc = subprocess.Popen([SGL_PY, "-m", "sglang.launch_server", "--model-path", model, "--host", "127.0.0.1",
                                 "--port", str(PORT), "--context-length", "8192", *advice["args"], *extra],
                                stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True, env=sgl_env)
        try:
            if not wait_healthy(proc, log):
                record("sglang", model, "(server start)", "FAIL", start_failure(log)); continue
            m = served(model, f"api:sglang/{model}")
            check_generation("sglang", model, m)
            if native:
                check_tools("sglang", model, m, "native", require_structured=True)
            else:
                check_tools("sglang", model, m, "prompt")
        finally:
            stop(proc)

## 5 · Cohere's hosted API → `api:`

Cohere serves an OpenAI-compatible endpoint. Tools are sent natively and come back as structured `tool_calls`.

In [ ]:
if RUN["cohere_api"]:
    for model in COHERE_API_MODELS:
        if not os.environ.get("CO_API_KEY"):
            skip("cohere api", model, "needs CO_API_KEY"); continue
        m = APIModel(model, api_base=COHERE_API_BASE, api_key=os.environ["CO_API_KEY"], name=f"api:cohere/{model}")
        check_generation("cohere api", model, m, retries=HOSTED)
        check_tools("cohere api", model, m, "native", require_structured=True, retries=HOSTED)

## 6 · Hugging Face Inference Providers → `api:`

HF's router is OpenAI-compatible, and `model:provider` picks the provider. One request probes availability first: a model the provider doesn't serve for this account is SKIP, not FAIL.

In [ ]:
if RUN["hf_router"]:
    model = HF_ROUTER_MODEL
    if not os.environ.get("HF_TOKEN"):
        skip("hf router", model, "needs HF_TOKEN")
    else:
        from auditkit.model import Request
        m = APIModel(model, api_base=HF_ROUTER_BASE, api_key=os.environ["HF_TOKEN"], name=f"api:hf/{model}")
        probe = m.generate([Request(prompt="Reply with OK.", params={"max_tokens": 8, "temperature": 0.0})])[0].completions[0]
        if probe.finish_reason == "error":
            skip("hf router", model, f"not served for this account: {str(probe.error)[:120]}")
        else:
            check_generation("hf router", model, m, retries=HOSTED)
            check_tools("hf router", model, m, "native", retries=HOSTED)

## 7 · Results

In [ ]:
df = pd.DataFrame(RESULTS)
pd.set_option("display.max_colwidth", 120)
display(df.pivot_table(index=["backend", "model"], columns="outcome", values="check", aggfunc="count", fill_value=0))
display(df[df.outcome == "FAIL"][["backend", "model", "check", "detail"]])
print("PASS", (df.outcome == "PASS").sum(), "| FAIL", (df.outcome == "FAIL").sum(), "| SKIP", (df.outcome == "SKIP").sum())

### Reading the table
- **FAIL:** a real problem. `detail` has the score, the errors and the first replies; server logs are in `/tmp/*.log`.
- **SKIP:** a missing token, not enough GPU memory, or a model the provider doesn't serve. It's never counted as a pass.
- **`tools (native) structured`:** the calls must come back as structured `tool_calls` from the server, not just be found in the reply text. On SGLang this is the check on the `cohere_command4` parser.
- **`native refused (no tool slot)`:** Tiny Aya has no tool slot in its template, so asking for native tools must raise a `CapabilityError`; accepting them would mean the tools were silently dropped.